In [117]:
import pandas as pd

surrogate_data = [
    {
        "sci_name": "Pan paniscus",
        "group": "Mammal",
        "category": "EN",
        "common_name": "Bonobo",
        "image_url": "https://upload.wikimedia.org/wikipedia/commons/3/32/Bonobo.jpg"
    },

    {
        "sci_name": "Gorilla gorilla gorilla",
        "group": "Mammal",
        "category": "CR",
        "common_name": "Western Lowland Gorilla",
        "image_url": "https://upload.wikimedia.org/wikipedia/commons/1/11/Western_lowland_gorilla.jpg"
    },

    {
        "sci_name": "Loxodonta cyclotis",
        "group": "Mammal",
        "category": "CR",
        "common_name": "Forest Elephant",
        "image_url": "https://upload.wikimedia.org/wikipedia/commons/6/63/African_forest_elephant.jpg"
    }
]

surrogates = pd.DataFrame(surrogate_data)

In [118]:
fiona.listlayers("data/Congobasin_perimeter/SPCB_Report_Perimeter.gdb")

['SPCB_Boundary']

In [119]:
congo_basin_boundries = gpd.read_file("data/Congobasin_perimeter/SPCB_Report_Perimeter.gdb", layer="SPCB_Boundary")
congo_basin_boundries.to_crs(epsg='4326')
congo_amphibians = gpd.read_file('congo_amphibians.gpkg').to_crs(epsg='4326')

In [120]:
surrogates = congo_amphibians[
    congo_amphibians["sci_name"].isin([
        "Amnirana albolabris","Leptopelis notatus","Leptopelis millsoni","Xenopus pygmaeus"
    ])
]

surrogates

,sci_name,presence,class,order_,family,genus,category,SHAPE_Leng,SHAPE_Area,geometry
99,Xenopus pygmaeus,1,AMPHIBIA,ANURA,PIPIDAE,Xenopus,LC,40.056897,60.956496,"MULTIPOLYGON (((29.58115 0.06191, 29.5545 0.05..."
251,Leptopelis notatus,1,AMPHIBIA,ANURA,ARTHROLEPTIDAE,Leptopelis,LC,68.111458,202.441154,"MULTIPOLYGON (((18.55943 -10.18965, 18.52747 -..."
336,Leptopelis millsoni,1,AMPHIBIA,ANURA,ARTHROLEPTIDAE,Leptopelis,LC,67.623560,140.182217,"MULTIPOLYGON (((9.63521 -2.41733, 9.63036 -2.4..."


In [121]:
# Amphibians biodiversity priority map. This map show where amphibians surrogate species overlap

#Create a grid over Congo Basin
xmin, ymin, xmax, ymax = congo_amphibians.total_bounds
cell_size = 0.25  # degrees (adjust later: 0.5, 0.25, etc.)

grid_cells = [
    box(x, y, x + cell_size, y + cell_size)
    for x in np.arange(xmin, xmax, cell_size)
    for y in np.arange(ymin, ymax, cell_size)
]

grid = gpd.GeoDataFrame(geometry=grid_cells, crs="EPSG:4326")

#Count surrogate occurrence per cell
grid["species"] = None

for i, cell in grid.iterrows():
    inside = surrogates[surrogates.intersects(cell.geometry)]
    grid.at[i, "species"] = list(inside["sci_name"].unique())
    
# Computer Richness
grid["richness"] = grid["species"].apply(len)
grid["species"] = grid["species"].apply(lambda x: ", ".join(x) if x else "")
#Color map in Folium
m = folium.Map(location=[0, 20], zoom_start=4, tiles="CartoDB positron")

colormap = cm.get_cmap("viridis")
max_richness = grid["richness"].max()

def get_color(r):

    if r == 0:
        return "transparent"

    norm = r / max_richness
    rgba = colormap(norm)

    # convert to hex
    return colors.rgb2hex(rgba)

def style_function(feature):

    r = feature["properties"]["richness"]

    if r == 0:
        return {
            "fillOpacity": 0,
            "weight": 0
        }

    return {
        "fillColor": get_color(r),
        "color": "black",
        "weight": 0.1,
        "fillOpacity": 0.7
    }

#Add to map
folium.GeoJson(
    grid,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["species","richness"], aliases=["Species present","Surrogate richness"])
).add_to(m)

#Add Legend
legend = bc.linear.viridis.scale(
    grid["richness"].min(),
    grid["richness"].max()
)

legend.caption = "Surrogate Species Richness"
legend.add_to(m)

m.save("amphibians_surrogate_richness_map.html")

C:\Users\vainq\AppData\Local\Temp\ipykernel_24688\2473967655.py:28: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("viridis")


In [122]:
# Amphibians biodiversity priority map. This map show where amphibians surrogate species overlap with other amphibians

#Create a grid over Congo Basin
xmin, ymin, xmax, ymax = congo_amphibians.total_bounds
cell_size = 0.25  # degrees (adjust later: 0.5, 0.25, etc.)

grid_cells = [
    box(x, y, x + cell_size, y + cell_size)
    for x in np.arange(xmin, xmax, cell_size)
    for y in np.arange(ymin, ymax, cell_size)
]

grid = gpd.GeoDataFrame(geometry=grid_cells, crs="EPSG:4326")

#Count surrogate occurrence per cell
grid["species"] = None
grid["surrogates"] = None

for i, cell in grid.iterrows():
    inside = congo_amphibians[congo_amphibians.intersects(cell.geometry)]
    grid.at[i, "species"] = list(inside["sci_name"].unique())

for i, cell in grid.iterrows():
    inside = surrogates[surrogates.intersects(cell.geometry)]
    grid.at[i, "surrogates"] = list(inside["sci_name"].unique())
# Computer Richness
grid["richness"] = grid["species"].apply(len)
grid["surrogates"] = grid["surrogates"].apply(lambda x: ", ".join(x) if x else "")
#Color map in Folium
m = folium.Map(location=[0, 20], zoom_start=4, tiles="CartoDB positron", control_scale=True)

colormap = cm.get_cmap("viridis")
max_richness = grid["richness"].max()

def get_color(r):

    if r == 0:
        return "transparent"

    norm = r / max_richness
    rgba = colormap(norm)

    # convert to hex
    return colors.rgb2hex(rgba)

def style_function(feature):

    r = feature["properties"]["richness"]

    if r == 0:
        return {
            "fillOpacity": 0,
            "weight": 0
        }

    return {
        "fillColor": get_color(r),
        "color": "black",
        "weight": 0.1,
        "fillOpacity": 0.7
    }

#Add to map
folium.GeoJson(
    grid,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["surrogates","richness"], aliases=["Surrogates present","Species richness"])
).add_to(m)

#Add Legend
legend = bc.linear.viridis.scale(
    grid["richness"].min(),
    grid["richness"].max()
)

legend.caption = "Surrogate Species Richness"
legend.add_to(m)

m.save("amphibians_richness_map.html")

C:\Users\vainq\AppData\Local\Temp\ipykernel_24688\135419543.py:32: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("viridis")
